In [1]:
import pandas as pd
pd.set_option('display.max_colwidth', 100)

In [2]:
df=pd.read_csv('sample_text.csv')
df.head()

,text,category
0,Meditation and yoga can improve mental health,Health
1,"Fruits, whole grains and vegetables helps control blood pressure",Health
2,These are the latest fashion trends for this week,Fashion
3,Vibrant color jeans for male are becoming a trend,Fashion
4,The concert starts at 7 PM tonight,Event


In [3]:
df.shape

(8, 2)

#### Step 1 : Create source embeddings for the text column

In [4]:
from sentence_transformers import SentenceTransformer

/home/surya/llm-projects/news_research_tool/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1226.35it/s]


In [6]:
sentences = [
    "The weather is lovely today.",
    "It's so sunny outside!",
    "He drove to the stadium.",
]

embeddings = model.encode(sentences)
print(embeddings.shape)

(3, 384)


In [7]:
vectors = model.encode(df["text"].tolist())
print(vectors.shape)

(8, 384)


In [8]:
dim = vectors.shape[1]
dim

384

#### Step 2 : Build a FAISS Index for vectors

In [9]:
import faiss

In [10]:
index = faiss.IndexFlatL2(dim)

#### Step 3 : Normalize the source vectors (as we are using L2 distance to measure similarity) and add to the index

In [11]:
index.add(vectors)

In [12]:
index

<faiss.swigfaiss.IndexFlatL2; proxy of <Swig Object of type 'faiss::IndexFlatL2 *' at 0x751a59a55df0> >

#### Step 4 : Encode search text using same encoder and normalize the output vector

In [13]:
search_query="I want to buy a polo t-shirt"
vec = model.encode(search_query)
vec.shape

(384,)

In [14]:
import numpy as np
svec =np.array(vec).reshape(1,-1)
svec.shape

(1, 384)

#### Step 5: Search for similar vector in the FAISS index created

In [23]:
index.search(svec, k=2)

(array([[1.4139274, 1.4256655]], dtype=float32), array([[3, 2]]))

In [20]:
distances, I = index.search(svec, k=2)

In [18]:
distances

array([[1.4139274, 1.4256655]], dtype=float32)

In [21]:
I

array([[3, 2]])

In [22]:
I.tolist()

[[3, 2]]

In [24]:
df.loc[I[0]]

,text,category
3,Vibrant color jeans for male are becoming a trend,Fashion
2,These are the latest fashion trends for this week,Fashion
